# Reply structure

**Among openings to the same OP, does structure help identify which reply earns the delta?** Compare three observable features of the original Markdown.

| Feature | Rule |
| --- | --- |
| Paragraph count | Count text blocks separated by blank lines. Consecutive list lines without blank lines can form one block. |
| Has a question mark | Flag a question mark in text with words after removing URLs and inline code. This includes rhetorical questions and other uses of the punctuation. |
| Has a Markdown list | Flag a line with a bullet or numbered-list marker followed by text. |

The opening extraction removes quoted lines and stops at marked edits before feature extraction. The rules skip fenced code and preserve list markers. These are text and formatting measurements. The archive can still contain unmarked edits. The [feature script](analyze_reply_structure.py) reads the training Parquet files and saves opening and pair features. The [model script](../models/train_p2_plus_reply_structure.py) evaluates the three features together with P2's existing inputs.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "team-setup").is_dir() and (path / "pyproject.toml").is_file()
)
ROOT = REPO / "project/data/processed/linguistic-features"
MODELS = ROOT / "paired-models"
MODEL_RUN_ID = "20261120T214728457174Z"
expanded_run = MODELS / "p2_plus_reply_structure" / MODEL_RUN_ID
expanded_report = json.loads((expanded_run / "results.json").read_text())
structure_run = ROOT / "reply-structure" / expanded_report["structure_input_run"]
report = json.loads((structure_run / "results.json").read_text())
assert report["split"] == expanded_report["split"] == "train"
assert hashlib.sha256((structure_run / "paired_scores.parquet").read_bytes()).hexdigest() == expanded_report["structure_input_sha256"]

pairs = pd.read_parquet(structure_run / "paired_scores.parquet")
openings = pd.read_parquet(structure_run / "opening_scores.parquet")
assert len(pairs) == report["pairs"]
assert pairs.groupby("post_id")["fold"].nunique().eq(1).all()
print(f"{len(pairs):,} training pairs across {report['discussions']:,} OP discussions")
print(f"{len(openings):,} distinct openings")

3,451 training pairs across 2,507 OP discussions
6,581 distinct openings


## Prevalence and examples

Each opening counts once. Inspect a seeded sample of flagged question and list lines to see what the rules capture.

In [2]:
paragraphs = openings["paragraph_count"]
display(paragraphs.describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(2).to_frame("Paragraph count"))
coverage = pd.DataFrame([
    {"Feature": label, "Openings": int(openings[feature].sum()), "Share (%)": 100 * openings[feature].mean()}
    for label, feature in [("Question mark", "has_question"), ("Markdown list", "has_list")]
]).set_index("Feature")
display(coverage.round(2))
examples = {}
for label, column in [("Question mark", "question_example"), ("Markdown list", "list_example")]:
    found = openings.loc[openings[column].ne(""), column]
    examples[label] = found.sample(min(4, len(found)), random_state=42).rename("Example")
display(pd.concat(examples).to_frame())

,Paragraph count
count,6581.00
mean,4.24
std,3.25
min,1.00
25%,2.00
50%,3.00
75%,5.00
95%,10.00
max,36.00


,Openings,Share (%)
Feature,,
Question mark,3017,45.84
Markdown list,549,8.34


Example
              comment_id                                                                                                                                                                                                                                        
Question mark t1_cf1avu5                                                                                                                    Do you begin to see the subtlety surrounding this topic? The philosophy, the mathematics, and the physics butt heads
              t1_cgl10xq                                                                                                                          How would it even be possible to divide the wealth equally as most wealth is in real assets such as buildings?
              t1_ckdfg3r  anything else that the husband paid for in good faith, considering it was her choice to leave her family in bad circumstances after living with them in bad circumstances?  In essence, she violated the contract to care for the kids
              t1_cmx7r0l  a ^^^^lie! ^^^^I ^^^^totally ^^^^care, ^^^^just ^^^^on ^^^^a ^^^^different ^^^^level!) . If you can't use that knowledge to describe yourself and others, what's the point? If and when you need to use what you gained in this class,
Markdown list t1_cgadifw                                            1. Sexual Objectification is the process of treating or representing a person as a sex object--that is, one who gives another sexual pleasure. (this is different from "sexual attraction.")
              t1_cket437                                                                                                                                 1) Except when they do spring a leak.  Calzones have a weak spot all around one side where the seam is.
              t1_cpiftnl                                                                                   * The rapid loss of species we are seeing today is estimated by experts to be between 1,000 and 10,000 times higher than the natural extinction rate.
              t1_cjsq1is                                                                  1) They have such a low point-of-view of the study of economics that they won't let me get a word in edgewise and I wish I had never given the person the time of day.

## Paired signal

Calculate the winner-higher rate among pairs where the feature differs. For binary indicators, this is the share won by the only opening with the feature. Ties are listed separately. Repeat the comparison for replies whose overall word counts differ by at most 50 words. Intervals resample OP discussions. Standalone models use the same three training folds.

In [3]:
labels = {
    "paragraph_count": "Paragraph count",
    "has_question": "Question mark",
    "has_list": "Markdown list",
}
rows = []
for feature, metric in report["metrics"].items():
    for subset, key in [("All pairs", "all_pairs"), ("Within 50 words", "within_50_words")]:
        counts = metric[key]
        low, high = counts["ci_95"]
        rows.append({
            "Feature": labels[feature], "Subset": subset, "Pairs": counts["pairs"],
            "Ties": counts["ties"], "Winner higher (%)": 100 * counts["winner_higher_rate"],
            "95% CI lower (%)": 100 * low, "95% CI upper (%)": 100 * high,
        })
association = pd.DataFrame(rows).set_index(["Feature", "Subset"])
display(association.round(2))
standalone = pd.DataFrame([
    {"Feature": labels[feature], "ROC-AUC": metric["roc_auc"], "Average precision": metric["average_precision"]}
    for feature, metric in report["metrics"].items()
]).set_index("Feature")
display(standalone.round(4))

Pairs  Ties  Winner higher (%)  95% CI lower (%)  95% CI upper (%)
Feature         Subset                                                                             
Paragraph count All pairs         3451   535              61.42             59.52             63.24
                Within 50 words   1144   256              53.49             50.33             56.77
Question mark   All pairs         3451  1811              53.78             51.21             56.32
                Within 50 words   1144   636              50.98             46.47             55.31
Markdown list   All pairs         3451  2985              60.73             56.31             65.08
                Within 50 words   1144  1048              53.12             43.29             63.54

,ROC-AUC,Average precision
Feature,,
Paragraph count,0.6301,0.6035
Question mark,0.5297,0.5158
Markdown list,0.5216,0.5139


## Incremental prediction beyond P2

Compare the archived P2 predictions with P2 plus all three structure features on identical validation rows. Both models use the three fixed OP-grouped folds and fit standardization inside each fold. The confidence intervals use 1,000 paired bootstrap draws by discussion. All results in this notebook use training cross-validation.

In [4]:
baseline_run = MODELS / "p2_word_count_and_content_word_zipf_differences" / expanded_report["baseline_run"]
base_report = json.loads((baseline_run / "results.json").read_text())
assert base_report["split"] == "train"
assert base_report["input_sha256"] == expanded_report["input_sha256"]
baseline = pd.read_parquet(baseline_run / "oof_predictions.parquet")
expanded = pd.read_parquet(expanded_run / "oof_predictions.parquet")
keys = ["pair_id", "post_id", "a_comment_id", "b_comment_id", "a_wins", "fold"]
pd.testing.assert_frame_equal(baseline[keys + base_report["features"]], expanded[keys + base_report["features"]])
pd.testing.assert_frame_equal(pairs[keys], expanded[keys])
added_features = [f"a_minus_b_{feature}" for feature in labels]
pd.testing.assert_frame_equal(pairs[added_features], expanded[added_features])
model_labels = ["P2", "P2 + reply structure"]
data = expanded.assign(**{
    model_labels[0]: baseline["oof_probability_a_wins"],
    model_labels[1]: expanded["oof_probability_a_wins"],
})

In [5]:
from sklearn.metrics import accuracy_score, average_precision_score, log_loss, roc_auc_score


def pairwise_accuracy(y, probability, sample_weight=None):
    return accuracy_score(y, probability >= 0.5, sample_weight=sample_weight)


metric_functions = {
    "ROC-AUC": roc_auc_score,
    "Average precision": average_precision_score,
    "Pairwise accuracy": pairwise_accuracy,
    "Log loss": log_loss,
}
y = data["a_wins"].to_numpy()
p2, expanded_probability = data[model_labels].to_numpy().T
post_codes, posts = pd.factorize(data["post_id"])
rng = np.random.default_rng(42)
differences = {name: [] for name in metric_functions}
for _ in range(1000):
    post_weights = rng.multinomial(len(posts), np.full(len(posts), 1 / len(posts)))
    weights = post_weights[post_codes]
    for name, metric in metric_functions.items():
        differences[name].append(
            metric(y, expanded_probability, sample_weight=weights)
            - metric(y, p2, sample_weight=weights)
        )

rows = []
for name, metric in metric_functions.items():
    base, extra = metric(y, p2), metric(y, expanded_probability)
    low, high = np.quantile(differences[name], [0.025, 0.975])
    rows.append({
        "Metric": name, "P2": base, "P2 + reply structure": extra, "Change": extra - base,
        "95% CI lower": low, "95% CI upper": high,
    })
performance = pd.DataFrame(rows).set_index("Metric")
display(performance.round(5))

,P2,P2 + reply structure,Change,95% CI lower,95% CI upper
Metric,,,,,
ROC-AUC,0.70324,0.70153,-0.00170,-0.00383,0.00026
Average precision,0.69426,0.69273,-0.00153,-0.00398,0.00105
Pairwise accuracy,0.64677,0.64590,-0.00087,-0.00609,0.00411
Log loss,0.62701,0.62801,0.00100,-0.00022,0.00231
